### ML4SCI deeplens gravitational lens finding- by Krrish Kumar

## LOADING THE DATA

In [ ]:
!pip install -q gdown
!gdown --id 1doUhVoq1-c9pamZVLpvjW1YRDMkKO1Q5 -O lens.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1doUhVoq1-c9pamZVLpvjW1YRDMkKO1Q5
From (redirected): https://drive.google.com/uc?id=1doUhVoq1-c9pamZVLpvjW1YRDMkKO1Q5&confirm=t&uuid=26cb23de-5f77-4343-9345-72ddeb940f67
To: /content/lens.zip
100% 2.11G/2.11G [00:28<00:00, 73.6MB/s]


In [ ]:
!unzip -q lens.zip -d lens_dataset

In [ ]:
import os

base_dir = "/content/lens_dataset"

train_lens = os.path.join(base_dir, "train_lenses")
train_nonlens = os.path.join(base_dir, "train_nonlenses")
test_lens = os.path.join(base_dir, "test_lenses")
test_nonlens = os.path.join(base_dir, "test_nonlenses")

print("Train Lens:", len(os.listdir(train_lens)))
print("Train Non-Lens:", len(os.listdir(train_nonlens)))
print("Test Lens:", len(os.listdir(test_lens)))
print("Test Non-Lens:", len(os.listdir(test_nonlens)))

Train Lens: 1730
Train Non-Lens: 28675
Test Lens: 195
Test Non-Lens: 19455


## EDA

If the distant source, the massive lens, and the observer are aligned in a perfectly straight line, the light is bent equally in all directions. This creates a perfect circle of light known as an Einstein Ring.

understanding the type of data


The dataset consists of normalized RGB images of size 64×64. Pixel values are already scaled to [0,1], eliminating the need for additional normalization. The relatively small spatial resolution suggests that fine structural distortions (e.g., arcs) must be captured efficiently through convolutional feature hierarchies rather than high-resolution detail.

In [ ]:
import numpy as np
import os
import random

sample_file = random.choice(os.listdir(train_lens))
sample_path = os.path.join(train_lens, sample_file)

img = np.load(sample_path)

print("Shape:", img.shape)
print("Data type:", img.dtype)
print("Min pixel:", img.min())
print("Max pixel:", img.max())

Shape: (3, 64, 64)
Data type: float32
Min pixel: 0.0
Max pixel: 1.0


In [ ]:
print("Example file:", os.listdir(train_lens)[0])
print("Shape:", np.load(os.path.join(train_lens, os.listdir(train_lens)[0])).shape)


Example file: 969.npy
Shape: (3, 64, 64)


calculating the imbalance in the dataset

The dataset exhibits significant class imbalance, with non-lens images heavily dominating the distribution. This imbalance introduces a risk of bias toward the majority class, potentially inflating accuracy while degrading recall for the lens class. Specialized strategies such as class weighting, focal loss, or balanced sampling will be required during model training.

In [ ]:
lens_count = len(os.listdir(train_lens))
nonlens_count = len(os.listdir(train_nonlens))

print("Imbalance Ratio (Non-lens : Lens) =", round(nonlens_count / lens_count, 2))

Imbalance Ratio (Non-lens : Lens) = 16.58


In [ ]:
import matplotlib.pyplot as plt

labels = ["Lens", "Non-Lens"]
counts = [lens_count, nonlens_count]

plt.figure(figsize=(6,4))
plt.bar(labels, counts)
plt.title("Training Set Class Distribution")
plt.ylabel("Number of Samples")
plt.show()

plotting random images